# 08 — Noisy-LJ latent-space rollout


In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from graph_utils import calc_p_ratio_rollout_sides

ROOT = next(path for path in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]] if (path/'src/lss').exists())
for path in [ROOT, ROOT/'src']:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from lss.graph import clone_graph
from lss.data import load_dataset
from lss.latent.experiment import run_latent_experiment, seed_everything
from lss.latent.simulation import r2_score
from lss.latent.training import decode_latent_to_graph
from lss.utils import resolve_device
from lss.plotting import PAPER_COLORS, apply_editorial_style, latent_dimension_color
from scripts.quick_lj_frozen_ae_propagator_sweep import encode_latent_table, evaluate, restore_ae
from scripts.train_lj_four_step_latent_propagator import fit as fit_four_step, raw_feature
from scripts.tune_lj_z3_delta_propagator import DeltaMLP

apply_editorial_style()
DEVICE = torch.device(resolve_device('auto'))
DEVICE

/home/alexz/Documents/course_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

## Configuration


In [ ]:
# These reproduce the Optuna study's split; the saved trial-025 AE is loaded below.
SEED = 6756435
MODEL_SEED = SEED + 10025  # Optuna base seed for trial 25.
# AE and propagator have separate training switches.
FORCE_TRAIN_AE = False 
FORCE_TRAIN_PROPAGATOR = False

# AE training configuration. Edit this dictionary before setting FORCE_TRAIN_AE=True.
AE_CONFIG = {
    'train_count': 300, 'val_count': 100,
    'latent_dim': 2, 'latent_tokens': 32, 'hidden_size': 64,
    'batch_graphs': 256, 'ae_max_train_frames_per_sim': 150,
    'ae_max_epochs': 20, 'ae_patience': 4,
    'ae_lr': 3.5e-4, 'ae_weight_decay': 1e-5,
    'node_feature_mode': 'normalized_delta',
    'ae_target_mode': 'normalized_delta',
}

# Dataset split and one-time latent encoding settings.
SPLIT_CONFIG = {
    'final_rollout_step': 199,  # 200 stored frames: indices 0 through 199
    # Fixed held-out validation/test regions, never used for fitting.
    'validation_start': 350, 'validation_count': 50,
    'test_start': 400, 'test_count': 100,
    'latent_encode_batch_size': 24,
}
REENCODE_LATENTS = False

OPTUNA_TRIAL = 25  # Best completed validation-rollout trial as of this notebook update.
DATA = ROOT/'data/lj-noisy-eps0.01-sigma1.0-cutoff1.122_500sims_200frames.pt'
OUTPUT = ROOT/'notebooks/results/08_history_aware_latent_rollout/lj_noisy/direct4_displacement_ae_rolling_latent_history'
OUTPUT.mkdir(parents=True, exist_ok=True)
AE_CACHE_IDENTITY = {'seed': SEED, 'ae': AE_CONFIG, 'model': 'single_stage_attention', 'edge_mode': 'stored', 'edge_multiplicity': 1, 'edge_vector_dim': 8}
AE_CACHE_TAG = hashlib.sha1(json.dumps(AE_CACHE_IDENTITY, sort_keys=True).encode()).hexdigest()[:12]
AE_CHECKPOINT = OUTPUT/f'displacement_only_ae_{AE_CACHE_TAG}.pt'
LATENT_CACHE = OUTPUT / (
    f"encoded_latents_{AE_CACHE_TAG}_frame{SPLIT_CONFIG['final_rollout_step']}_"
    f"val{SPLIT_CONFIG['validation_start']}-{SPLIT_CONFIG['validation_count']}_"
    f"test{SPLIT_CONFIG['test_start']}-{SPLIT_CONFIG['test_count']}.pt"
)
AE_BUNDLE = OUTPUT/f'ae_bundle_{AE_CACHE_TAG}.pt'

if SPLIT_CONFIG['validation_start'] < AE_CONFIG['train_count']:
    raise ValueError('validation_start must be at least train_count.')
if SPLIT_CONFIG['test_start'] < SPLIT_CONFIG['validation_start'] + SPLIT_CONFIG['validation_count']:
    raise ValueError('test_start must be after the complete validation region.')
if SPLIT_CONFIG['test_start'] + SPLIT_CONFIG['test_count'] > 500:
    raise ValueError('Configured split exceeds the 500-trajectory dataset.')
if FORCE_TRAIN_AE and not FORCE_TRAIN_PROPAGATOR:
    raise ValueError('A retrained AE has a new latent coordinate system; also set FORCE_TRAIN_PROPAGATOR=True.')
print({
    'device': str(DEVICE), 'optuna_trial': OPTUNA_TRIAL, 'AE': str(AE_CHECKPOINT), 'data': str(DATA),
    'ae_config': AE_CONFIG,
    'split_config': SPLIT_CONFIG,
})

{'device': 'cuda', 'optuna_trial': 25, 'AE': '/home/alexz/Documents/course_project/notebooks/results/08_history_aware_latent_rollout/lj_noisy/direct4_displacement_ae_rolling_latent_history/displacement_only_ae_e1fa41661fd5.pt', 'data': '/home/alexz/Documents/course_project/data/lj-noisy-eps0.01-sigma1.0-cutoff1.122_500sims_200frames.pt', 'ae_config': {'train_count': 300, 'val_count': 100, 'latent_dim': 2, 'latent_tokens': 32, 'hidden_size': 64, 'batch_graphs': 256, 'ae_max_train_frames_per_sim': 150, 'ae_max_epochs': 20, 'ae_patience': 4, 'ae_lr': 0.00035, 'ae_weight_decay': 1e-05, 'node_feature_mode': 'normalized_delta', 'ae_target_mode': 'normalized_delta'}, 'split_config': {'final_rollout_step': 199, 'validation_start': 350, 'validation_count': 50, 'test_start': 400, 'test_count': 100, 'latent_encode_batch_size': 24}}


## Autoencoder


In [ ]:
# Match the AE's shared one-edge-per-undirected-pair dataset loader.
all_sims = load_dataset(DATA, edge_multiplicity=1, edge_vector_dim=8)
TRAIN_AE_NOW = FORCE_TRAIN_AE or not AE_CHECKPOINT.exists()
if TRAIN_AE_NOW:
    ae_cfg = {
        'dataset_name': 'lj_noisy', 'split_seed': SEED, 'model_seed': SEED,
        'device': str(DEVICE), 'split_stratify_temperature': False,
        'min_train_p_ratio': None, 'pos_dim': 2, 'frame_skip': 1,
        'train_frame_start_order': 0, 'edge_multiplicity': 1,
        'edge_vector_dim': 8, 'edge_mode': 'stored',
        'autoencoder_model': 'single_stage_attention', 'early_stop_min_delta': 1e-5,
        'should_rollout': False, 'should_train_propagator': False,
        'cache_path': str(AE_BUNDLE), 'force_train': True, **AE_CONFIG,
    }
    source = {**ae_cfg, 'source_name': 'Noisy LJ', 'label': 'Noisy LJ | displacement-only AE', 'path': str(DATA)}
    seed_everything(SEED)
    ae_result = run_latent_experiment(source, ae_cfg, device=DEVICE)
    ae, normalizers, ae_params = ae_result['ae'], ae_result['normalizers'], ae_result['params']
    torch.save({
        'params': ae_params, 'ae_state_dict': {key: value.detach().cpu() for key, value in ae.state_dict().items()},
        'normalizers': {key: value.detach().cpu() for key, value in normalizers.items()},
    }, AE_CHECKPOINT)
    print(f'saved trained AE: {AE_CHECKPOINT}')
else:
    ae, normalizers, ae_params = restore_ae(AE_CHECKPOINT, DEVICE)

for parameter in ae.parameters():
    parameter.requires_grad_(False)
ae.eval()
generator = torch.Generator().manual_seed(int(ae_params['split_seed']))
order = torch.randperm(len(all_sims), generator=generator).tolist()
# Use the saved AE's split, not editable AE_CONFIG, when the AE is frozen.
ae_train_count = int(ae_params['train_count'])
train_sims = [all_sims[index] for index in order[:ae_train_count]]
val_start = SPLIT_CONFIG['validation_start']
test_start = SPLIT_CONFIG['test_start']
val_sims = [all_sims[index] for index in order[val_start:val_start + SPLIT_CONFIG['validation_count']]]
test_sims = [all_sims[index] for index in order[test_start:test_start + SPLIT_CONFIG['test_count']]]
split_ids = [set(order[:ae_train_count]), set(order[val_start:val_start + SPLIT_CONFIG['validation_count']]), set(order[test_start:test_start + SPLIT_CONFIG['test_count']])]
assert not (split_ids[0] & split_ids[1] or split_ids[0] & split_ids[2] or split_ids[1] & split_ids[2]), 'Training, validation, and test trajectories must be disjoint.'
sims = train_sims + val_sims + test_sims

MAX_EVAL_STEP = min(SPLIT_CONFIG['final_rollout_step'], min(len(sim) - 1 for sim in sims))
expected_shape = (len(sims), MAX_EVAL_STEP + 1, int(ae_params['latent_dim']))
if LATENT_CACHE.exists() and not (REENCODE_LATENTS or TRAIN_AE_NOW):
    latent_table = torch.load(LATENT_CACHE, map_location='cpu', weights_only=True)
    if tuple(latent_table.shape) != expected_shape:
        raise ValueError(f'Latent cache has shape {tuple(latent_table.shape)}, expected {expected_shape}. Set REENCODE_LATENTS=True.')
    print(f'loaded latent cache: {LATENT_CACHE}')
else:
    latent_table = encode_latent_table(
        ae, normalizers, sims, max_frame=MAX_EVAL_STEP,
        batch_size=SPLIT_CONFIG['latent_encode_batch_size'], device=DEVICE,
        node_feature_mode=ae_params['node_feature_mode'],
    )
    torch.save(latent_table, LATENT_CACHE)

train_stop = len(train_sims)
val_stop = train_stop + len(val_sims)
train_z = latent_table[:train_stop]
val_z = latent_table[train_stop:val_stop]
test_z = latent_table[val_stop:]
print({'train': tuple(train_z.shape), 'validation': tuple(val_z.shape), 'test': tuple(test_z.shape)})

loaded latent cache: /home/alexz/Documents/course_project/notebooks/results/08_history_aware_latent_rollout/lj_noisy/direct4_displacement_ae_rolling_latent_history/encoded_latents_e1fa41661fd5_frame199_val350-50_test400-100.pt
{'train': (300, 200, 2), 'validation': (50, 200, 2), 'test': (100, 200, 2)}


## Propagator


In [ ]:
# Best Optuna propagator configuration; keep the AE above frozen.
PROPAGATOR_CONFIG = {
    'target_step': 100,
    'unroll_steps': 1,
    'hidden_size': 64, 'depth': 2,
    'learning_rate': 2e-4, 'weight_decay': 1e-5,
    'max_epochs': 50, 'early_stop_patience': 12,
    'batch_size': 256, 'early_stop_min_delta': 1e-6,
    'rollout_eval_every': 2,
    'history_mode': 'fixed_latent_frames',
    'checkpoint_metric': 'val_rollout_p_ratio_r2',
}
PROPAGATOR_CONFIG['observed_frames'] = (1, 10)
PROPAGATOR_CACHE_TAG = hashlib.sha1(json.dumps({'ae': AE_CACHE_TAG, 'propagator': PROPAGATOR_CONFIG}, sort_keys=True).encode()).hexdigest()[:12]
PROPAGATOR_SEEDS = [MODEL_SEED, MODEL_SEED + 1000, MODEL_SEED + 2000]
PROPAGATOR_CHECKPOINTS = {seed: OUTPUT/f'best_propagator_{PROPAGATOR_CACHE_TAG}_seed{seed}.pt' for seed in PROPAGATOR_SEEDS}
PROPAGATOR_CHECKPOINT = PROPAGATOR_CHECKPOINTS[PROPAGATOR_SEEDS[0]]
assert tuple(sorted(set(PROPAGATOR_CONFIG['observed_frames']))) == tuple(PROPAGATOR_CONFIG['observed_frames'])
assert 0 <= min(PROPAGATOR_CONFIG['observed_frames']) < max(PROPAGATOR_CONFIG['observed_frames']) < PROPAGATOR_CONFIG['target_step']
assert 1 <= PROPAGATOR_CONFIG['unroll_steps'] < PROPAGATOR_CONFIG['target_step'] - 2
assert PROPAGATOR_CONFIG['target_step'] <= MAX_EVAL_STEP
print(PROPAGATOR_CONFIG)

In [ ]:
# Reload so this cell always reflects edits to the standalone propagator helper.
import importlib
import scripts.train_lj_four_step_latent_propagator as latent_propagator_helper
latent_propagator_helper = importlib.reload(latent_propagator_helper)
fit_four_step = latent_propagator_helper.fit

def fit_propagator(config, seed):
    return fit_four_step(
        ae, normalizers, train_z, val_z, val_sims, seed=seed, device=DEVICE,
        rollout_eval_every=config['rollout_eval_every'],
        hidden_size=config['hidden_size'], depth=config['depth'],
        learning_rate=config['learning_rate'], weight_decay=config['weight_decay'],
        unroll_steps=config['unroll_steps'], target_step=config['target_step'],
        max_epochs=config['max_epochs'], early_stop_patience=config['early_stop_patience'],
        batch_size=config['batch_size'], early_stop_min_delta=config['early_stop_min_delta'],
        history_mode=config['history_mode'], ae_target_mode=ae_params['ae_target_mode'],
        observed_frames=config['observed_frames'], checkpoint_metric=config['checkpoint_metric'],
    )

TRAIN_PROPAGATOR_NOW = FORCE_TRAIN_PROPAGATOR or TRAIN_AE_NOW or not PROPAGATOR_CHECKPOINT.exists()
if TRAIN_PROPAGATOR_NOW:
    model, prop_stats, best = fit_propagator(PROPAGATOR_CONFIG, MODEL_SEED)
    torch.save({
        'model_state_dict': {key: value.detach().cpu() for key, value in model.state_dict().items()},
        'normalization': {key: value.detach().cpu() for key, value in prop_stats.items()},
        'configuration': {'input': ['z_current', 'observed_latent_pair', 'progress'], 'target': 'delta_z', **PROPAGATOR_CONFIG},
        'best_epoch': best['epoch'], 'val_terminal_latent_mse': best['loss'],
        'checkpoint_metric': best['checkpoint_metric'], 'best_checkpoint_score': best['selection_score'],
        'val_decoded_field_mse': best.get('field_mse', np.nan),
        'history': best['history'], 'ae_checkpoint': str(AE_CHECKPOINT),
    }, PROPAGATOR_CHECKPOINT)
else:
    saved = torch.load(PROPAGATOR_CHECKPOINT, map_location='cpu', weights_only=False)
    latent_dim = int(ae_params['latent_dim'])
    saved_config = saved.get('configuration', {})
    saved_config = {**saved_config, 'observed_frames': tuple(saved_config['observed_frames'])}
    checked_keys = ['target_step', 'unroll_steps', 'hidden_size', 'depth', 'history_mode', 'observed_frames', 'checkpoint_metric']
    mismatches = {key: (PROPAGATOR_CONFIG[key], saved_config.get(key)) for key in checked_keys if PROPAGATOR_CONFIG[key] != saved_config.get(key)}
    if mismatches:
        raise ValueError(f'PROPAGATOR_CONFIG does not match the loaded checkpoint: {mismatches}. Set FORCE_TRAIN_PROPAGATOR=True to train this new configuration.')
    if saved_config.get('history_mode') in {'fixed_latent_frames', 'fixed_latent_frames3'}:
        feature_latents = 1 + len(saved_config['observed_frames'])
    else:
        feature_latents = 4 if saved_config.get('history_mode') in {'rolling_latents3_anchor', 'fixed_observed_latents3', 'fixed_spaced_latents3'} else 3
    model = DeltaMLP(
        feature_latents * latent_dim + 1, latent_dim,
        int(saved_config.get('hidden_size', PROPAGATOR_CONFIG['hidden_size'])),
        int(saved_config.get('depth', PROPAGATOR_CONFIG['depth'])),
    ).to(DEVICE)
    model.load_state_dict(saved['model_state_dict'])
    prop_stats = {key: value.to(DEVICE) for key, value in saved['normalization'].items()}
    best = {'epoch': saved.get('best_epoch'), 'loss': saved.get('val_terminal_latent_mse', saved.get('val_decoded_field_mse')), 'field_mse': saved.get('val_decoded_field_mse'), 'checkpoint_metric': saved.get('checkpoint_metric'), 'selection_score': saved.get('best_checkpoint_score'), 'history': saved.get('history', [])}

model.eval()
print({'best_epoch': best['epoch'], 'checkpoint_metric': best.get('checkpoint_metric'), 'best_checkpoint_score': best.get('selection_score'), 'validation_terminal_latent_mse': best['loss'], 'validation_decoded_field_mse': best.get('field_mse'), 'checkpoint': str(PROPAGATOR_CHECKPOINT)})
prop_history = pd.DataFrame(best['history'])
if len(prop_history):
    display(prop_history.round(6))
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), constrained_layout=True)
    if 'train_multistep_loss' in prop_history:
        axes[0].plot(prop_history.epoch, prop_history.train_multistep_loss, marker='o')
    else:
        axes[0].text(.5, .5, 'Not stored in this older checkpoint', ha='center', transform=axes[0].transAxes)
    axes[0].set(xlabel='Epoch', ylabel='Training multistep loss')
    axes[1].plot(prop_history.epoch, prop_history.val_terminal_latent_mse, marker='o')
    axes[1].set(xlabel='Epoch', ylabel='Validation terminal latent MSE')
    p_ratio_history = prop_history.dropna(subset=['val_rollout_p_ratio_r2'])
    axes[2].plot(p_ratio_history.epoch, p_ratio_history.val_rollout_p_ratio_r2, marker='o')
    axes[2].set(xlabel='Epoch', ylabel='Validation rollout p-ratio R²')
    plt.show()

def save_seeded_propagator(path, trained_model, trained_stats, trained_best):
    torch.save({
        'model_state_dict': {key: value.detach().cpu() for key, value in trained_model.state_dict().items()},
        'normalization': {key: value.detach().cpu() for key, value in trained_stats.items()},
        'configuration': {'input': ['z_current', 'observed_latent_pair', 'progress'], 'target': 'delta_z', **PROPAGATOR_CONFIG},
        'best_epoch': trained_best['epoch'], 'val_terminal_latent_mse': trained_best['loss'],
        'checkpoint_metric': trained_best['checkpoint_metric'], 'best_checkpoint_score': trained_best['selection_score'],
        'val_decoded_field_mse': trained_best.get('field_mse', np.nan), 'history': trained_best['history'],
        'ae_checkpoint': str(AE_CHECKPOINT),
    }, path)

def load_seeded_propagator(path):
    saved = torch.load(path, map_location='cpu', weights_only=False)
    saved_config = saved['configuration']; latent_dim = int(ae_params['latent_dim'])
    loaded_model = DeltaMLP((1 + len(saved_config['observed_frames']))*latent_dim + 1, latent_dim, int(saved_config['hidden_size']), int(saved_config['depth'])).to(DEVICE)
    loaded_model.load_state_dict(saved['model_state_dict']); loaded_model.eval()
    loaded_stats = {key: value.to(DEVICE) for key, value in saved['normalization'].items()}
    loaded_best = {'epoch': saved.get('best_epoch'), 'loss': saved.get('val_terminal_latent_mse'), 'field_mse': saved.get('val_decoded_field_mse'), 'checkpoint_metric': saved.get('checkpoint_metric'), 'selection_score': saved.get('best_checkpoint_score'), 'history': saved.get('history', [])}
    return loaded_model, loaded_stats, loaded_best

propagator_runs = {PROPAGATOR_SEEDS[0]: (model, prop_stats, best)}
for run_seed in PROPAGATOR_SEEDS[1:]:
    checkpoint = PROPAGATOR_CHECKPOINTS[run_seed]
    if FORCE_TRAIN_PROPAGATOR or TRAIN_AE_NOW or not checkpoint.exists():
        run_model, run_stats, run_best = fit_propagator(PROPAGATOR_CONFIG, run_seed)
        save_seeded_propagator(checkpoint, run_model, run_stats, run_best)
    else:
        run_model, run_stats, run_best = load_seeded_propagator(checkpoint)
    propagator_runs[run_seed] = (run_model, run_stats, run_best)
print({seed: {'best_epoch': run_best['epoch'], 'validation_score': run_best.get('selection_score')} for seed, (_, _, run_best) in propagator_runs.items()})

## Main 8D frame-100 scatter


In [ ]:
MAIN_LATENT_DIM = 8
MAIN_SCATTER_STEP = 100
MAIN_8D_OUTPUT = ROOT/'notebooks'/'results'/'08b_lj_8d_propagator_and_latent_analysis'
MAIN_8D_PAIRS = MAIN_8D_OUTPUT/'test_p_ratio_pairs.csv'
if not MAIN_8D_PAIRS.exists():
    raise FileNotFoundError('Run 08b once to create the finalized held-out 8D evaluation.')
main_pairs = pd.read_csv(MAIN_8D_PAIRS)
main_pairs = main_pairs[main_pairs.step.eq(MAIN_SCATTER_STEP)].replace([np.inf, -np.inf], np.nan).dropna(subset=['true_p_ratio', 'pred_p_ratio'])
main_summary = []
for measurement, frame in main_pairs.groupby('measurement', sort=False):
    true = frame.true_p_ratio.to_numpy(float); pred = frame.pred_p_ratio.to_numpy(float)
    main_summary.append({'measurement': measurement, 'latent_dim': MAIN_LATENT_DIM,
                         'p_ratio_r2': r2_score(true, pred),
                         'p_ratio_pearson': float(np.corrcoef(true, pred)[0, 1]), 'n': len(frame)})
display(pd.DataFrame(main_summary).set_index('measurement').round(4))


In [ ]:
def scatter_pratio(ax, frame, title):
    true = frame.true_p_ratio.to_numpy(float); pred = frame.pred_p_ratio.to_numpy(float)
    score = r2_score(true, pred); pearson = float(np.corrcoef(true, pred)[0, 1])
    lo, hi = min(true.min(), pred.min()), max(true.max(), pred.max())
    pad = 0.05 * max(hi - lo, 1e-6)
    ax.scatter(true, pred, s=30, alpha=.72, edgecolor='none')
    ax.plot([lo-pad, hi+pad], [lo-pad, hi+pad], '--', color='black', lw=1)
    ax.set(xlabel='True p-ratio', ylabel='Predicted p-ratio')
    ax.text(.04, .96, f'{title}\nR² = {score:.3f}\nr = {pearson:.3f}\nN = {len(true)}', transform=ax.transAxes, va='top')

print(f'Noisy-LJ AE ceiling and latent-rollout p-ratio predictions at frame {MAIN_SCATTER_STEP}.')
fig, axes = plt.subplots(1, 2, figsize=(10.6, 5.0), constrained_layout=True)
for ax, measurement in zip(axes, ['AE ceiling', 'Selected propagator']):
    scatter_pratio(ax, main_pairs[main_pairs.measurement.eq(measurement)], f'8D {measurement}')
fig.savefig(OUTPUT/f'p_ratio_scatter_8d_ae_and_rollout_frame{MAIN_SCATTER_STEP}.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT/f'p_ratio_scatter_8d_ae_and_rollout_frame{MAIN_SCATTER_STEP}.pdf', bbox_inches='tight')
plt.show()

rollout_pairs_8d = main_pairs[main_pairs.measurement.eq('Selected propagator')].copy()
print(f'Noisy-LJ latent-rollout p-ratio predictions at frame {MAIN_SCATTER_STEP}.')
fig, ax = plt.subplots(figsize=(5.4, 5.0), constrained_layout=True)
scatter_pratio(ax, rollout_pairs_8d, '8D noisy-LJ latent rollout')
fig.savefig(OUTPUT/f'p_ratio_scatter_8d_rollout_frame{MAIN_SCATTER_STEP}.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT/f'p_ratio_scatter_8d_rollout_frame{MAIN_SCATTER_STEP}.pdf', bbox_inches='tight')
plt.show()
rollout_pairs_8d.to_csv(OUTPUT/f'p_ratio_scatter_8d_rollout_frame{MAIN_SCATTER_STEP}.csv', index=False)


## Legacy 2D frame-100 diagnostic


In [ ]:
# Reload the feature helper independently of the training cell.
import importlib
import scripts.train_lj_four_step_latent_propagator as latent_propagator_helper
latent_propagator_helper = importlib.reload(latent_propagator_helper)
def rollout_to_step(model, stats, z, target_step):
    return latent_propagator_helper.rollout(
        model, stats, z, DEVICE, target_step, PROPAGATOR_CONFIG['history_mode'],
        progress_scale=PROPAGATOR_CONFIG['target_step'],
        observed_frames=PROPAGATOR_CONFIG['observed_frames'],
    ).cpu()

def p_ratio_pairs(sims, predicted_z, step):
    rows = []
    with torch.no_grad():
        for sim_index, (sim, latent) in enumerate(zip(sims, predicted_z)):
            graph = decode_latent_to_graph(
                ae, sim, latent.to(DEVICE), step, pos_dim=2,
                ae_target_mode=ae_params.get('ae_target_mode', 'normalized_delta'), normalizers=normalizers,
                device=DEVICE,
            ).cpu()
            rows.append({
                'sim_index': sim_index,
                'true_p_ratio': float(calc_p_ratio_rollout_sides(sim, step)),
                'pred_p_ratio': float(calc_p_ratio_rollout_sides([clone_graph(sim[0]).cpu(), graph], -1)),
            })
    return pd.DataFrame(rows)

target_step = PROPAGATOR_CONFIG['target_step']
predicted_test_z = rollout_to_step(model, prop_stats, test_z, target_step)
ae_metrics = evaluate(
    ae, normalizers, test_sims, test_z[:, target_step], target_step, DEVICE,
    ae_target_mode=ae_params['ae_target_mode'],
)
rollout_metrics = evaluate(
    ae, normalizers, test_sims, predicted_test_z, target_step, DEVICE,
    ae_target_mode=ae_params['ae_target_mode'],
)
display(pd.DataFrame([
    {'measurement': 'AE reconstruction ceiling', **ae_metrics},
    {'measurement': f"{PROPAGATOR_CONFIG['unroll_steps']}-step latent rollout", **rollout_metrics},
]).set_index('measurement').round(4))

In [ ]:
rollout_pairs = p_ratio_pairs(test_sims, predicted_test_z, PROPAGATOR_CONFIG['target_step'])
true = rollout_pairs.true_p_ratio.to_numpy()
pred = rollout_pairs.pred_p_ratio.to_numpy()
score = r2_score(true, pred)
pearson = float(np.corrcoef(true, pred)[0, 1])
lo, hi = min(true.min(), pred.min()), max(true.max(), pred.max())
pad = 0.05 * max(hi - lo, 1e-6)

print(f"Noisy-LJ rollout p-ratio predictions at frame {PROPAGATOR_CONFIG['target_step']}.")
fig, ax = plt.subplots(figsize=(5.4, 5.0), constrained_layout=True)
ax.scatter(true, pred, s=30, alpha=.72, edgecolor='none')
ax.plot([lo-pad, hi+pad], [lo-pad, hi+pad], '--', color='black', lw=1)
ax.set(xlabel='True p-ratio', ylabel='Predicted p-ratio')
ax.text(.04, .96, f'R² = {score:.3f}\nr = {pearson:.3f}\nN = {len(true)}', transform=ax.transAxes, va='top')
fig.savefig(OUTPUT/f"p_ratio_scatter_frame{PROPAGATOR_CONFIG['target_step']}.png", dpi=220, bbox_inches='tight')
plt.show()
rollout_pairs.to_csv(OUTPUT/f"p_ratio_scatter_frame{PROPAGATOR_CONFIG['target_step']}.csv", index=False)

## Rollout R² by frame


In [ ]:
curve_rows = []
curve_steps = [10, 25, 50, 75, 100, 125, 150, 175, MAX_EVAL_STEP]
for step in dict.fromkeys(curve_steps):
    ceiling_pairs = p_ratio_pairs(test_sims, test_z[:, step], step)
    curve_rows.append({'step': step, 'measurement': 'AE ceiling', 'seed': np.nan, 'p_ratio_r2': r2_score(ceiling_pairs.true_p_ratio, ceiling_pairs.pred_p_ratio), 'p_ratio_pearson': float(np.corrcoef(ceiling_pairs.true_p_ratio, ceiling_pairs.pred_p_ratio)[0,1])})
    for run_seed, (run_model, run_stats, _) in propagator_runs.items():
        predicted_z = rollout_to_step(run_model, run_stats, test_z, step)
        pairs = p_ratio_pairs(test_sims, predicted_z, step)
        curve_rows.append({'step': step, 'measurement': f"{PROPAGATOR_CONFIG['unroll_steps']}-step rollout", 'seed': run_seed, 'p_ratio_r2': r2_score(pairs.true_p_ratio, pairs.pred_p_ratio), 'p_ratio_pearson': float(np.corrcoef(pairs.true_p_ratio, pairs.pred_p_ratio)[0,1])})
curve = pd.DataFrame(curve_rows)
display(curve.round(4))

print('Frozen-AE noisy-LJ rollout over three independently trained propagator seeds: mean p-ratio R² ± one standard deviation.')
fig, ax = plt.subplots(figsize=(6.8, 4.4), constrained_layout=True)
ceiling = curve[curve.measurement.eq('AE ceiling')].sort_values('step')
ax.plot(ceiling.step, ceiling.p_ratio_r2.clip(lower=0), marker='o', lw=1.8, color=PAPER_COLORS['slate'], linestyle=':', label='AE ceiling')
rollouts = curve[curve.measurement.ne('AE ceiling')]
for _, seed_group in rollouts.groupby('seed'):
    seed_group = seed_group.sort_values('step'); ax.plot(seed_group.step, seed_group.p_ratio_r2.clip(lower=0), color=PAPER_COLORS['blue'], lw=.8, alpha=.24)
summary = rollouts.groupby('step').p_ratio_r2.agg(['mean','std']).reset_index().sort_values('step')
ax.fill_between(summary.step, (summary['mean']-summary['std']).clip(lower=0), (summary['mean']+summary['std']).clip(lower=0), color=PAPER_COLORS['blue'], alpha=.18, linewidth=0)
ax.plot(summary.step, summary['mean'].clip(lower=0), marker='s', lw=2, color=PAPER_COLORS['blue'], label='latent propagator mean')
ax.axhline(0, color='black', lw=.8)
ax.axvline(PROPAGATOR_CONFIG['target_step'], color='gray', ls='--', lw=1, label='training horizon')
ax.set(xlabel='Rollout frame', ylabel='p-ratio R²')
ax.legend(frameon=False)
fig.savefig(OUTPUT/'p_ratio_r2_vs_step_to_final_frame.png', dpi=220, bbox_inches='tight')
plt.show()
curve.to_csv(OUTPUT/'p_ratio_r2_vs_step_to_final_frame.csv', index=False)

## Latent dimension: AE ceiling and rollout


In [ ]:
LATENT_DIMENSIONS = [1, 2, 4, 8]
DIMENSION_CURVE_STEPS = [20, 50, 75, 100, 125, 150, 175, MAX_EVAL_STEP]
# The top-level force switches also apply to every dimension in this sweep.
FORCE_DIMENSION_AE_TRAIN = FORCE_TRAIN_AE
REENCODE_DIMENSION_LATENTS = FORCE_DIMENSION_AE_TRAIN
FORCE_DIMENSION_PROPAGATOR_TRAIN = FORCE_TRAIN_PROPAGATOR
DIMENSION_SWEEP_OUTPUT = OUTPUT/'latent_dimension_comparison'/f'{AE_CACHE_TAG}_{PROPAGATOR_CACHE_TAG}'
DIMENSION_SWEEP_OUTPUT.mkdir(parents=True, exist_ok=True)
print({'latent_dimensions': LATENT_DIMENSIONS, 'curve_steps': DIMENSION_CURVE_STEPS, 'observed_frames': PROPAGATOR_CONFIG['observed_frames'], 'unroll_steps': PROPAGATOR_CONFIG['unroll_steps']})


In [ ]:
def prepare_dimension_autoencoder(latent_dim):
    if latent_dim == int(ae_params['latent_dim']) and int(ae_params['train_count']) == len(train_sims):
        return ae, normalizers, ae_params, train_z, val_z, test_z
    case_output = DIMENSION_SWEEP_OUTPUT/f'latent_{latent_dim}'
    case_output.mkdir(parents=True, exist_ok=True)
    bundle_path = case_output/'ae_bundle.pt'
    latent_path = case_output/f'encoded_latents_frame{MAX_EVAL_STEP}.pt'
    dimension_config = {
        'dataset_name': 'lj_noisy', 'split_seed': SEED, 'model_seed': SEED + latent_dim,
        'device': str(DEVICE), 'split_stratify_temperature': False, 'min_train_p_ratio': None,
        'pos_dim': 2, 'frame_skip': 1, 'train_frame_start_order': 0, 'edge_multiplicity': 1,
        'edge_vector_dim': 8, 'edge_mode': 'stored', 'autoencoder_model': 'single_stage_attention',
        'early_stop_min_delta': 1e-5, 'should_rollout': False, 'should_train_propagator': False,
        'cache_path': str(bundle_path), 'force_train': FORCE_DIMENSION_AE_TRAIN,
        **AE_CONFIG, 'latent_dim': int(latent_dim),
    }
    source = {**dimension_config, 'source_name': 'Noisy LJ', 'label': f'Noisy LJ | {latent_dim}D displacement AE', 'path': str(DATA)}
    seed_everything(SEED + latent_dim)
    dimension_result = run_latent_experiment(source, dimension_config, device=DEVICE)
    dimension_ae = dimension_result['ae']; dimension_normalizers = dimension_result['normalizers']; dimension_params = dimension_result['params']
    for parameter in dimension_ae.parameters():
        parameter.requires_grad_(False)
    dimension_ae.eval()
    # Do not rely on the notebook-global `sims`: it can be stale after partial reruns.
    dimension_sims = train_sims + val_sims + test_sims
    expected = (len(dimension_sims), MAX_EVAL_STEP + 1, int(latent_dim))
    if latent_path.exists() and not (REENCODE_DIMENSION_LATENTS or FORCE_DIMENSION_AE_TRAIN):
        dimension_z = torch.load(latent_path, map_location='cpu', weights_only=True)
        if tuple(dimension_z.shape) != expected:
            print(f'Re-encoding stale latent cache: {latent_path} has {tuple(dimension_z.shape)}, expected {expected}.')
            dimension_z = encode_latent_table(dimension_ae, dimension_normalizers, dimension_sims, max_frame=MAX_EVAL_STEP, batch_size=SPLIT_CONFIG['latent_encode_batch_size'], device=DEVICE, node_feature_mode=dimension_params['node_feature_mode'])
            torch.save(dimension_z, latent_path)
    else:
        dimension_z = encode_latent_table(dimension_ae, dimension_normalizers, dimension_sims, max_frame=MAX_EVAL_STEP, batch_size=SPLIT_CONFIG['latent_encode_batch_size'], device=DEVICE, node_feature_mode=dimension_params['node_feature_mode'])
        torch.save(dimension_z, latent_path)
    return dimension_ae, dimension_normalizers, dimension_params, dimension_z[:train_stop], dimension_z[train_stop:val_stop], dimension_z[val_stop:]

curve_csv = DIMENSION_SWEEP_OUTPUT/'ae_ceiling_and_rollout_r2_by_dimension_and_step.csv'
required_columns = {'latent_dim', 'step', 'measurement', 'p_ratio_r2'}
if curve_csv.exists() and not FORCE_DIMENSION_PROPAGATOR_TRAIN:
    cached_curve = pd.read_csv(curve_csv)
    curve_rows = cached_curve.to_dict('records') if required_columns.issubset(cached_curve.columns) else []
else:
    curve_rows = []

def has_complete_curve(rows, latent_dim):
    table = pd.DataFrame(rows)
    if table.empty:
        return False
    subset = table[table.latent_dim.eq(latent_dim)]
    return set(DIMENSION_CURVE_STEPS).issubset(set(subset.step)) and {'AE ceiling', 'Latent rollout'}.issubset(set(subset.measurement))

for latent_dim in LATENT_DIMENSIONS:
    if has_complete_curve(curve_rows, latent_dim) and not FORCE_DIMENSION_PROPAGATOR_TRAIN:
        print(f'{latent_dim}D curve loaded from CSV', flush=True)
        continue
    dim_ae, dim_normalizers, dim_params, dim_train_z, dim_val_z, dim_test_z = prepare_dimension_autoencoder(latent_dim)
    model_dim, stats_dim, best_dim = fit_four_step(
        dim_ae, dim_normalizers, dim_train_z, dim_val_z, val_sims, seed=MODEL_SEED, device=DEVICE,
        rollout_eval_every=PROPAGATOR_CONFIG['rollout_eval_every'], hidden_size=PROPAGATOR_CONFIG['hidden_size'], depth=PROPAGATOR_CONFIG['depth'],
        learning_rate=PROPAGATOR_CONFIG['learning_rate'], weight_decay=PROPAGATOR_CONFIG['weight_decay'],
        unroll_steps=PROPAGATOR_CONFIG['unroll_steps'], target_step=PROPAGATOR_CONFIG['target_step'],
        max_epochs=PROPAGATOR_CONFIG['max_epochs'], early_stop_patience=PROPAGATOR_CONFIG['early_stop_patience'],
        batch_size=PROPAGATOR_CONFIG['batch_size'], early_stop_min_delta=PROPAGATOR_CONFIG['early_stop_min_delta'],
        history_mode=PROPAGATOR_CONFIG['history_mode'], ae_target_mode=dim_params['ae_target_mode'], observed_frames=PROPAGATOR_CONFIG['observed_frames'], checkpoint_metric=PROPAGATOR_CONFIG['checkpoint_metric'],
    )
    dim_rows = []
    for step in DIMENSION_CURVE_STEPS:
        predicted_z = latent_propagator_helper.rollout(model_dim, stats_dim, dim_test_z, DEVICE, step, PROPAGATOR_CONFIG['history_mode'], progress_scale=PROPAGATOR_CONFIG['target_step'], observed_frames=PROPAGATOR_CONFIG['observed_frames']).cpu()
        ae_metrics = evaluate(dim_ae, dim_normalizers, test_sims, dim_test_z[:, step], step, DEVICE, ae_target_mode=dim_params['ae_target_mode'])
        rollout_metrics = evaluate(dim_ae, dim_normalizers, test_sims, predicted_z, step, DEVICE, ae_target_mode=dim_params['ae_target_mode'])
        dim_rows.extend([
            {'latent_dim': latent_dim, 'step': step, 'measurement': 'AE ceiling', 'p_ratio_r2': ae_metrics['p_ratio_r2'], 'p_ratio_pearson': ae_metrics['p_ratio_pearson'], 'best_epoch': best_dim['epoch']},
            {'latent_dim': latent_dim, 'step': step, 'measurement': 'Latent rollout', 'p_ratio_r2': rollout_metrics['p_ratio_r2'], 'p_ratio_pearson': rollout_metrics['p_ratio_pearson'], 'best_epoch': best_dim['epoch']},
        ])
    curve_rows = [row for row in curve_rows if int(row['latent_dim']) != latent_dim] + dim_rows
    pd.DataFrame(curve_rows).to_csv(curve_csv, index=False)
dimension_curve = pd.DataFrame(curve_rows).sort_values(['latent_dim', 'measurement', 'step']).reset_index(drop=True)
display(dimension_curve.round(4))


In [ ]:
print('Noisy-LJ AE ceiling and latent rollout by latent dimension.')
fig, ax = plt.subplots(figsize=(8.0, 4.8), constrained_layout=True)
for latent_dim in LATENT_DIMENSIONS:
    subset = dimension_curve[dimension_curve.latent_dim.eq(latent_dim)]
    color = latent_dimension_color(latent_dim)
    ceiling = subset[subset.measurement.eq('AE ceiling')].sort_values('step')
    rollout = subset[subset.measurement.eq('Latent rollout')].sort_values('step')
    ax.plot(ceiling.step, ceiling.p_ratio_r2.clip(lower=0), color=color, ls=':', lw=1.8, label=f'{latent_dim}D AE ceiling')
    ax.plot(rollout.step, rollout.p_ratio_r2.clip(lower=0), color=color, marker='o', ms=3.8, lw=1.8, label=f'{latent_dim}D rollout')
ax.axhline(0, color='0.2', lw=.8)
ax.axvline(PROPAGATOR_CONFIG['target_step'], color='0.45', ls='--', lw=1, label='training horizon')
ax.set(xlabel='Rollout frame', ylabel='Held-out test p-ratio R²')
ax.legend(frameon=False, ncol=2)
fig.savefig(DIMENSION_SWEEP_OUTPUT/'ae_ceiling_and_rollout_r2_by_dimension_and_step.png', dpi=300, bbox_inches='tight')
fig.savefig(DIMENSION_SWEEP_OUTPUT/'ae_ceiling_and_rollout_r2_by_dimension_and_step.pdf', bbox_inches='tight')
plt.show()


## Latent dimension versus observed-history spacing


In [ ]:
HISTORY_K_VALUES = list(range(2, 21))
FORCE_HISTORY_SWEEP_TRAIN = FORCE_TRAIN_PROPAGATOR
HISTORY_SWEEP_OUTPUT = OUTPUT/'latent_dimension_history_sweep'/f'{AE_CACHE_TAG}_{PROPAGATOR_CACHE_TAG}'
HISTORY_SWEEP_OUTPUT.mkdir(parents=True, exist_ok=True)
history_csv = HISTORY_SWEEP_OUTPUT/'frame100_history_sweep_test.csv'
required_history_columns = {'latent_dim', 'history_start', 'k', 'split', 'p_ratio_r2'}
if history_csv.exists() and not FORCE_HISTORY_SWEEP_TRAIN:
    cached_history = pd.read_csv(history_csv)
    history_rows = cached_history.to_dict('records') if required_history_columns.issubset(cached_history.columns) else []
else:
    history_rows = []

for latent_dim in LATENT_DIMENSIONS:
    dim_ae, dim_normalizers, dim_params, dim_train_z, dim_val_z, dim_test_z = prepare_dimension_autoencoder(latent_dim)
    for k in HISTORY_K_VALUES:
        already_done = any(int(row['latent_dim']) == latent_dim and int(row['history_start']) == 1 and int(row['k']) == k for row in history_rows)
        if already_done and not FORCE_HISTORY_SWEEP_TRAIN:
            continue
        observed_frames = (1, int(k))
        model_history, stats_history, best_history = fit_four_step(
            dim_ae, dim_normalizers, dim_train_z, dim_val_z, val_sims, seed=MODEL_SEED, device=DEVICE,
            rollout_eval_every=PROPAGATOR_CONFIG['rollout_eval_every'], hidden_size=PROPAGATOR_CONFIG['hidden_size'], depth=PROPAGATOR_CONFIG['depth'],
            learning_rate=PROPAGATOR_CONFIG['learning_rate'], weight_decay=PROPAGATOR_CONFIG['weight_decay'],
            unroll_steps=PROPAGATOR_CONFIG['unroll_steps'], target_step=PROPAGATOR_CONFIG['target_step'],
            max_epochs=PROPAGATOR_CONFIG['max_epochs'], early_stop_patience=PROPAGATOR_CONFIG['early_stop_patience'],
            batch_size=PROPAGATOR_CONFIG['batch_size'], early_stop_min_delta=PROPAGATOR_CONFIG['early_stop_min_delta'],
            history_mode='fixed_latent_frames', ae_target_mode=dim_params['ae_target_mode'], observed_frames=observed_frames, checkpoint_metric=PROPAGATOR_CONFIG['checkpoint_metric'],
        )
        predicted_z = latent_propagator_helper.rollout(model_history, stats_history, dim_test_z, DEVICE, PROPAGATOR_CONFIG['target_step'], 'fixed_latent_frames', progress_scale=PROPAGATOR_CONFIG['target_step'], observed_frames=observed_frames).cpu()
        metrics = evaluate(dim_ae, dim_normalizers, test_sims, predicted_z, PROPAGATOR_CONFIG['target_step'], DEVICE, ae_target_mode=dim_params['ae_target_mode'])
        history_rows = [row for row in history_rows if not (int(row['latent_dim']) == latent_dim and int(row['history_start']) == 1 and int(row['k']) == k)]
        history_rows.append({'latent_dim': latent_dim, 'history_start': 1, 'k': k, 'split': 'test', 'best_epoch': best_history['epoch'], 'terminal_latent_mse': best_history['loss'], **metrics})
        pd.DataFrame(history_rows).to_csv(history_csv, index=False)
history_sweep = pd.DataFrame(history_rows)
display(history_sweep.sort_values(['latent_dim', 'k']).round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 4.8), constrained_layout=True)
for latent_dim in LATENT_DIMENSIONS:
    subset = history_sweep[(history_sweep.latent_dim.eq(latent_dim)) & (history_sweep.history_start.eq(1))].sort_values('k')
    ax.plot(subset.k, subset.p_ratio_r2.clip(lower=0), color=latent_dimension_color(latent_dim), marker='o', ms=3.8, lw=1.8, label=f'{latent_dim}D')
ax.axhline(0, color='0.2', lw=.8)
ax.set_xticks(HISTORY_K_VALUES[::2])
ax.set(xlabel='Second observed latent frame k in (1, k)', ylabel='Held-out test p-ratio R²')
ax.legend(frameon=False)
fig.savefig(HISTORY_SWEEP_OUTPUT/'frame100_test_rollout_r2_vs_history_by_latent_dimension.png', dpi=300, bbox_inches='tight')
fig.savefig(HISTORY_SWEEP_OUTPUT/'frame100_test_rollout_r2_vs_history_by_latent_dimension.pdf', bbox_inches='tight')
plt.show()
